# Answer Generator Fine-Tuning — T5 / FLAN-T5

Fine-tunes a seq2seq model on Dataset B (`text -> ideal_response`) so the chatbot
can generate fluent, grounded educational answers.

**Run on Colab / Kaggle GPU.** Compares `t5-small` and `google/flan-t5-base`.

1. Install deps
2. Upload `Dataset_B_SRKI.csv` (or `SU_final_250k_B.csv`)
3. Fine-tune, evaluate, save & optionally push to Hugging Face Hub.

> **⚠️ Data caveat:** In the provided datasets, `ideal_response` is templated
> filler that does not actually answer the question, so a model trained on
> `text → ideal_response` will learn that same filler. Use this notebook to learn
> conversational style, OR first replace `ideal_response` with real answers
> (from the scraped site / an LLM) for factual quality. The deployed app grounds
> factual answers on scraped web content regardless.

In [ ]:
!pip -q install "transformers>=4.38" "datasets>=2.16" "accelerate>=0.27" evaluate rouge-score sentencepiece pandas huggingface-hub

In [ ]:
DATA_CSV = "Dataset_B_SRKI.csv"   # or SU_final_250k_B.csv
INSTITUTION = "srki"
MODEL = "google/flan-t5-base"     # or 't5-small' for a lighter/faster model
NUM_EPOCHS = 3
MAX_IN, MAX_OUT = 128, 128
BATCH = 8
MAX_ROWS = None                   # e.g. 60000 for a fast SU run
PUSH_TO_HUB = False
HF_USER = "your-hf-username"

In [ ]:
from google.colab import files
import os
if not os.path.exists(DATA_CSV):
    files.upload()

In [ ]:
import pandas as pd, torch
from datasets import Dataset
from sklearn.model_selection import train_test_split

df = pd.read_csv(DATA_CSV, dtype=str, keep_default_na=False)
df.columns = [c.strip().lower() for c in df.columns]
df = df[(df['text'].str.len() > 0) & (df['ideal_response'].str.len() > 0)]
if MAX_ROWS:
    df = df.sample(min(len(df), MAX_ROWS), random_state=42)
PREFIX = 'answer the educational question: '
df['input_text'] = PREFIX + df['text']
df['target_text'] = df['ideal_response']
train_df, val_df = train_test_split(df, test_size=0.1, random_state=42)
print('train', len(train_df), 'val', len(val_df))
train_ds = Dataset.from_pandas(train_df[['input_text', 'target_text']], preserve_index=False)
val_ds = Dataset.from_pandas(val_df[['input_text', 'target_text']], preserve_index=False)

In [ ]:
from transformers import (AutoTokenizer, AutoModelForSeq2SeqLM,
                          Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq)

tok = AutoTokenizer.from_pretrained(MODEL)
def preprocess(b):
    model_in = tok(b['input_text'], max_length=MAX_IN, truncation=True)
    labels = tok(text_target=b['target_text'], max_length=MAX_OUT, truncation=True)
    model_in['labels'] = labels['input_ids']
    return model_in
train_t = train_ds.map(preprocess, batched=True, remove_columns=train_ds.column_names)
val_t = val_ds.map(preprocess, batched=True, remove_columns=val_ds.column_names)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL)

In [ ]:
save_dir = f'model_{INSTITUTION}_generator'
args = Seq2SeqTrainingArguments(
    output_dir='gen_out', num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH, per_device_eval_batch_size=BATCH,
    learning_rate=3e-4, weight_decay=0.01, eval_strategy='epoch', save_strategy='epoch',
    load_best_model_at_end=True, predict_with_generate=True,
    fp16=torch.cuda.is_available(), logging_steps=200, report_to='none')
trainer = Seq2SeqTrainer(model=model, args=args, train_dataset=train_t, eval_dataset=val_t,
                         tokenizer=tok, data_collator=DataCollatorForSeq2Seq(tok, model=model))
trainer.train()
trainer.save_model(save_dir); tok.save_pretrained(save_dir)
print('Saved to', save_dir)

In [ ]:
# quick sanity check
from transformers import pipeline
gen = pipeline('text2text-generation', model=save_dir, tokenizer=tok)
for q in ['How can I apply for admission in MCA?', 'What is the fee structure for MSc IT?']:
    print(q, '->', gen(PREFIX + q, max_new_tokens=80)[0]['generated_text'])

In [ ]:
import shutil
shutil.make_archive(save_dir, 'zip', save_dir)
print('Zipped:', save_dir + '.zip')
if PUSH_TO_HUB:
    from huggingface_hub import login; login()
    repo = f'{HF_USER}/{INSTITUTION}-generator'
    model.push_to_hub(repo); tok.push_to_hub(repo)
    print('Pushed to', repo)
    print(f'Set in .env:  {INSTITUTION.upper()}_GENERATOR_MODEL={repo}  and  USE_GENERATOR=true')